In [1]:
# ==============================================================================
# FASE 3: PREPARAÇÃO DOS DADOS (CRISP-DM - Itens 3.1 a 3.4)
# ==============================================================================
import pandas as pd
import numpy as np

df_imoveis = pd.read_csv("../data/raw/imoveis.csv")
df_bairros = pd.read_csv("../data/raw/indicadores_bairros.csv")

print(f"Registros brutos iniciais: {len(df_imoveis)} linhas x {df_imoveis.shape[1]} colunas")

# ------------------------------------------------------------------------------
# 3.1 Seleção (Filtragem de atributos confidenciais e irrelevantes)
# ------------------------------------------------------------------------------
# Removemos 'cpf_proprietario' (dado sensível/LGPD) e 'id_imovel' (identificador sem poder preditivo)
df = df_imoveis.drop(columns=["cpf_proprietario", "id_imovel"]).copy()
print("\n=== 3.1 SELEÇÃO ===")
print("Colunas removidas: ['cpf_proprietario', 'id_imovel']")

# ------------------------------------------------------------------------------
# 3.2 Limpeza (Duplicatas, texto, outliers e valores ausentes)
# ------------------------------------------------------------------------------
print("\n=== 3.2 LIMPEZA ===")

# 1. Remover duplicatas exatas
antes = len(df)
df = df.drop_duplicates()
print(f"1) Duplicatas removidas: {antes - len(df)}")

# 2. Padronizar texto da coluna 'bairro' ("asa sul" -> "Asa Sul")
df["bairro"] = df["bairro"].str.strip().replace({"asa sul": "Asa Sul"})
print(f"2) Bairros padronizados ({df['bairro'].nunique()} bairros únicos): {sorted(df['bairro'].unique())}")

# 3. Remover outliers / erros de digitação absurdos
antes = len(df)
df = df[
    (df["area_m2"] > 10) & (df["area_m2"] <= 2000) &
    (df["preco_venda"] >= 50000) & (df["preco_venda"] <= 10000000) &
    (df["quartos"] <= 15)
]
print(f"3) Outliers/erros de digitação removidos: {antes - len(df)}")

# 4. Tratamento de valores ausentes (imputação)
nulos_antes = df.isnull().sum().sum()
df["vagas_garagem"] = df["vagas_garagem"].fillna(df["vagas_garagem"].median())
df["idade_imovel"] = df["idade_imovel"].fillna(df["idade_imovel"].median())
df["condominio"] = df["condominio"].fillna(df.groupby("tipo")["condominio"].transform("median"))
print(f"4) Valores nulos tratados: {nulos_antes} -> {df.isnull().sum().sum()} restantes")

# ------------------------------------------------------------------------------
# 3.4 Integração (Mesclagem da base de imóveis com indicadores dos bairros)
# ------------------------------------------------------------------------------
df = df.merge(df_bairros, on="bairro", how="left")
print("\n=== 3.4 INTEGRAÇÃO ===")
print(f"Base integrada com 'indicadores_bairros.csv' via chave 'bairro'. Novo formato: {df.shape}")

# ------------------------------------------------------------------------------
# 3.3 Construção (Feature Engineering - novos atributos derivados)
# ------------------------------------------------------------------------------
df["area_por_quarto"] = (df["area_m2"] / df["quartos"]).round(2)
df["total_comodos"] = df["quartos"] + df["banheiros"]
df["relacao_condominio_m2"] = (df["condominio"] / df["area_m2"]).round(2)

print("\n=== 3.3 CONSTRUÇÃO DE ATRIBUTOS ===")
print("Novos atributos criados: ['area_por_quarto', 'total_comodos', 'relacao_condominio_m2']")

# ------------------------------------------------------------------------------
# Salvamento da base tratada para a Fase 4 (Modelagem)
# ------------------------------------------------------------------------------
df.to_csv("../data/processed/imoveis_tratado.csv", index=False)
print(f"\nSUCESSO! Base final salva em '../data/processed/imoveis_tratado.csv'")
print(f"Dimensão final: {df.shape[0]} linhas x {df.shape[1]} colunas")
display(df.head())

Registros brutos iniciais: 1215 linhas x 11 colunas

=== 3.1 SELEÇÃO ===
Colunas removidas: ['cpf_proprietario', 'id_imovel']

=== 3.2 LIMPEZA ===
1) Duplicatas removidas: 15
2) Bairros padronizados (8 bairros únicos): ['Asa Norte', 'Asa Sul', 'Guará', 'Planaltina', 'Samambaia', 'Sobradinho', 'Taguatinga', 'Águas Claras']
3) Outliers/erros de digitação removidos: 5
4) Valores nulos tratados: 102 -> 0 restantes

=== 3.4 INTEGRAÇÃO ===
Base integrada com 'indicadores_bairros.csv' via chave 'bairro'. Novo formato: (1195, 12)

=== 3.3 CONSTRUÇÃO DE ATRIBUTOS ===
Novos atributos criados: ['area_por_quarto', 'total_comodos', 'relacao_condominio_m2']

SUCESSO! Base final salva em '../data/processed/imoveis_tratado.csv'
Dimensão final: 1195 linhas x 15 colunas


,bairro,tipo,area_m2,quartos,banheiros,vagas_garagem,idade_imovel,condominio,preco_venda,renda_media_bairro,indice_infraestrutura,preco_m2_ref,area_por_quarto,total_comodos,relacao_condominio_m2
0,Águas Claras,Casa,149.1,3,3,2.0,37.0,0.00,1008700.0,8200,8.5,7800,49.70,6,0.00
1,Sobradinho,Apartamento,88.0,3,3,2.0,11.0,804.41,535300.0,5400,7.5,5500,29.33,6,9.14
2,Samambaia,Apartamento,58.7,2,2,1.0,4.0,579.59,229600.0,3200,6.1,3800,29.35,4,9.87
3,Taguatinga,Apartamento,53.8,2,2,1.0,42.0,666.15,231400.0,4800,7.2,5200,26.90,4,12.38
4,Asa Norte,Apartamento,54.3,2,2,1.0,14.0,508.73,563100.0,11800,9.2,11000,27.15,4,9.37
